Cell 1: Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("Libraries loaded successfully.")

Cell 2: Paths

In [ ]:
RAW = "data/raw"
PROCESSED = "data/processed"

logon_path   = os.path.join(RAW, "logon.csv")
file_path    = os.path.join(RAW, "file.csv")
email_path   = os.path.join(RAW, "email.csv")
device_path  = os.path.join(RAW, "device.csv")
http_path    = os.path.join(RAW, "http.csv")
psycho_path  = os.path.join(RAW, "psychometric.csv")

print("Paths defined.")

Cell 3: Load small files only

In [ ]:
logon_df  = pd.read_csv(logon_path)
file_df   = pd.read_csv(file_path)
email_df  = pd.read_csv(email_path)
device_df = pd.read_csv(device_path)

print("Shapes:")
print(f"  Logon:  {logon_df.shape}")
print(f"  File:   {file_df.shape}")
print(f"  Email:  {email_df.shape}")
print(f"  Device: {device_df.shape}")

Cell 4: HTTP in chunks

In [ ]:
def aggregate_http_chunk(chunk):
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
    chunk['day'] = chunk['date'].dt.date
    chunk = chunk.dropna(subset=['user', 'day'])
    return chunk.groupby(['user', 'day']).size().reset_index(name='http_count')

http_parts = []

for i, chunk in enumerate(pd.read_csv(http_path, usecols=['user', 'date'], chunksize=100000)):
    part = aggregate_http_chunk(chunk)
    http_parts.append(part)
    print(f"Chunk {i+1}: raw={len(chunk)}, agg={len(part)}")

http_agg = pd.concat(http_parts, ignore_index=True)
http_agg = http_agg.groupby(['user', 'day'], as_index=False)['http_count'].sum()

print("HTTP aggregation complete:", http_agg.shape)
print(http_agg.head())

Cell 5: Merge all aggregated data

In [ ]:
def to_user_day(df, date_col='date'):
    out = df[['user', date_col]].copy()
    out[date_col] = pd.to_datetime(out[date_col], errors='coerce')
    out = out.dropna(subset=['user', date_col])
    out['day'] = out[date_col].dt.normalize()
    return out[['user', 'day']]

# Base user-day index from all available logs
all_user_days = pd.concat([
    to_user_day(logon_df),
    to_user_day(file_df),
    to_user_day(email_df),
    to_user_day(device_df),
    http_agg.assign(day=pd.to_datetime(http_agg['day'], errors='coerce'))[['user', 'day']]
], ignore_index=True).drop_duplicates().sort_values(['user', 'day']).reset_index(drop=True)

# Logon features
logon_tmp = logon_df.copy()
logon_tmp['date'] = pd.to_datetime(logon_tmp['date'], errors='coerce')
logon_tmp = logon_tmp.dropna(subset=['user', 'date'])
logon_tmp['day'] = logon_tmp['date'].dt.normalize()
logon_features = logon_tmp.groupby(['user', 'day']).size().reset_index(name='logon_count')

# File features
file_tmp = file_df.copy()
file_tmp['date'] = pd.to_datetime(file_tmp['date'], errors='coerce')
file_tmp = file_tmp.dropna(subset=['user', 'date'])
file_tmp['day'] = file_tmp['date'].dt.normalize()
file_features = file_tmp.groupby(['user', 'day']).size().reset_index(name='file_count')

# Email features
email_tmp = email_df.copy()
email_tmp['date'] = pd.to_datetime(email_tmp['date'], errors='coerce')
email_tmp = email_tmp.dropna(subset=['user', 'date'])
email_tmp['day'] = email_tmp['date'].dt.normalize()
email_features = email_tmp.groupby(['user', 'day']).size().reset_index(name='email_count')

# Device features
device_tmp = device_df.copy()
device_tmp['date'] = pd.to_datetime(device_tmp['date'], errors='coerce')
device_tmp = device_tmp.dropna(subset=['user', 'date'])
device_tmp['day'] = device_tmp['date'].dt.normalize()
device_features = device_tmp.groupby(['user', 'day']).size().reset_index(name='device_count')

# Ensure HTTP day is datetime64 and ready for merge
http_features = http_agg.copy()
http_features['day'] = pd.to_datetime(http_features['day'], errors='coerce')
http_features = http_features.dropna(subset=['user', 'day'])

# Merge all user-day counts
user_day_df = (
    all_user_days
    .merge(logon_features, on=['user', 'day'], how='left')
    .merge(file_features, on=['user', 'day'], how='left')
    .merge(email_features, on=['user', 'day'], how='left')
    .merge(device_features, on=['user', 'day'], how='left')
    .merge(http_features, on=['user', 'day'], how='left')
    .fillna(0)
)

count_cols = ['logon_count', 'file_count', 'email_count', 'device_count', 'http_count']
user_day_df[count_cols] = user_day_df[count_cols].astype('int32')

print("User-day feature table shape:", user_day_df.shape)
print(user_day_df.head())

# Optional cleanup to free RAM before next cell
del logon_tmp, file_tmp, email_tmp, device_tmp
del logon_features, file_features, email_features, device_features, http_features, all_user_days

Cell 6: Load monthly LDAP

In [ ]:
# Cell 6: load monthly LDAP snapshots, keep latest record per user_id, merge into user_day_df

import glob
import os

ldap_files = sorted(glob.glob(os.path.join("data", "raw", "ldap", "*.csv")))
print(f"Found {len(ldap_files)} LDAP files")

ldap_parts = []

for f in ldap_files:
    temp = pd.read_csv(f)
    temp["snapshot_file"] = os.path.basename(f)
    ldap_parts.append(temp)
    print(f"{os.path.basename(f)} -> {temp.shape}")

ldap_all = pd.concat(ldap_parts, ignore_index=True)
print("Combined LDAP shape:", ldap_all.shape)
print("LDAP columns:", ldap_all.columns.tolist())

ldap_all["snapshot_month"] = pd.to_datetime(
    ldap_all["snapshot_file"].str.extract(r"(\d{4}-\d{2})")[0],
    format="%Y-%m",
    errors="coerce"
)

candidate_cols = [
    "user_id", "employee_name", "email", "role", "projects", "business_unit",
    "functional_unit", "department", "team", "supervisor"
]
keep_cols = [c for c in candidate_cols if c in ldap_all.columns]

ldap_all = ldap_all[["snapshot_month"] + keep_cols].copy()
ldap_all = ldap_all.dropna(subset=["user_id"])

ldap_latest = (
    ldap_all.sort_values(["user_id", "snapshot_month"])
    .drop_duplicates(subset=["user_id"], keep="last")
    .reset_index(drop=True)
)

user_day_df = user_day_df.merge(
    ldap_latest.drop(columns=["snapshot_month"]),
    left_on="user",
    right_on="user_id",
    how="left"
)

user_day_df = user_day_df.drop(columns=["user_id"])

print("Latest LDAP shape:", ldap_latest.shape)
print("User-day + LDAP shape:", user_day_df.shape)
print(user_day_df.head())
print("\nTop missing values:")
print(user_day_df.isna().sum().sort_values(ascending=False).head(15))

Cell 5:  Save checkpoint after building user_day_df

In [ ]:
# Cell 5A: save checkpoint after building user_day_df

processed_path = os.path.join("data", "processed")
# os.makedirs(processed_path, exist_ok=True)

user_day_path = os.path.join(processed_path, "user_day_df.parquet")
user_day_df.to_parquet(user_day_path, index=False)

print(f"Saved: {user_day_path}")
print("Shape:", user_day_df.shape)

Cell 6: Inspect one unlabeled r6.2 file safely

In [ ]:
# Cell 6: inspect one unlabeled r6.2 file safely

r62_1_path = os.path.join("data", "raw", "r6.2-1.csv")

# sample = pd.read_csv(r62_1_path, header=None, nrows=5)
sample = pd.read_csv(r62_1_path, engine='python', on_bad_lines='warn')
print("Shape:", sample.shape)
print("Columns:", sample.shape[1])
print(sample.head())

# show raw row lengths if needed
print("\nRow lengths:")
print(sample.apply(lambda r: r.notna().sum(), axis=1).tolist())

Cell 7: Parse r6.2-1 through r6.2-5

In [ ]:
# Cell 7: normalize r6.2-1 through r6.2-5 into one activity table

import csv
from pathlib import Path

file_user_map = {
    "r6.2-1.csv": "ACM2278",
    "r6.2-2.csv": "CMP2946",
    "r6.2-3.csv": "PLJ1771",
    "r6.2-4.csv": "CDE1846",
    "r6.2-5.csv": "MBG3183",
}

common_cols = [
    "activity_type", "id", "date", "user", "pc",
    "field_1", "field_2", "field_3", "field_4", "field_5",
    "field_6", "field_7", "field_8", "field_9", "field_10",
    "source_file", "insider_user"
]

all_parts = []

for fname, fixed_user in file_user_map.items():
    path = Path("data/raw") / fname
    rows = []

    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if row:
                rows.append(row)

    print(f"\n{fname}: {len(rows)} raw rows")

    normalized = []
    for row in rows:
        act = row[0]
        out = {c: None for c in common_cols}
        out["activity_type"] = act
        out["source_file"] = fname
        out["insider_user"] = fixed_user

        if act == "logon":
            # logon,id,date,user,pc,activity
            vals = row + [None] * (6 - len(row))
            out.update({
                "activity_type": vals[0],
                "id": vals[1],
                "date": vals[2],
                "user": vals[3],
                "pc": vals[4],
                "field_1": vals[5],   # activity
            })

        elif act == "device":
            # device,id,date,user,pc,file_tree,activity
            vals = row + [None] * (7 - len(row))
            out.update({
                "activity_type": vals[0],
                "id": vals[1],
                "date": vals[2],
                "user": vals[3],
                "pc": vals[4],
                "field_1": vals[5],   # file_tree
                "field_2": vals[6],   # activity
            })

        elif act == "http":
            # http,id,date,user,pc,url,activity,content
            vals = row + [None] * (8 - len(row))
            out.update({
                "activity_type": vals[0],
                "id": vals[1],
                "date": vals[2],
                "user": vals[3],
                "pc": vals[4],
                "field_1": vals[5],   # url
                "field_2": vals[6],   # activity
                "field_3": vals[7],   # content
            })

        elif act == "email":
            # email,id,date,user,pc,to,cc,bcc,from,activity,size,attachments,content
            vals = row + [None] * (13 - len(row))
            out.update({
                "activity_type": vals[0],
                "id": vals[1],
                "date": vals[2],
                "user": vals[3],
                "pc": vals[4],
                "field_1": vals[5],   # to
                "field_2": vals[6],   # cc
                "field_3": vals[7],   # bcc
                "field_4": vals[8],   # from
                "field_5": vals[9],   # activity
                "field_6": vals[10],  # size
                "field_7": vals[11],  # attachments
                "field_8": vals[12],  # content
            })

        elif act == "file":
            # file,id,date,user,pc,filename,activity,to_removable_media,from_removable_media,content_label,content
            vals = row + [None] * (11 - len(row))
            out.update({
                "activity_type": vals[0],
                "id": vals[1],
                "date": vals[2],
                "user": vals[3],
                "pc": vals[4],
                "field_1": vals[5],   # filename
                "field_2": vals[6],   # activity
                "field_3": vals[7],   # to_removable_media
                "field_4": vals[8],   # from_removable_media
                "field_5": vals[9],   # content_label
                "field_6": vals[10],  # content
            })

        normalized.append(out)

    df = pd.DataFrame(normalized)
    all_parts.append(df)

    print(df["activity_type"].value_counts(dropna=False).to_dict())
    print(df.head(2))

activity_df = pd.concat(all_parts, ignore_index=True)
activity_df["date"] = pd.to_datetime(activity_df["date"], errors="coerce")
activity_df["day"] = activity_df["date"].dt.normalize()

print("\nCombined activity shape:", activity_df.shape)
print(activity_df[["activity_type", "user", "date", "day"]].head())

Cell 8: Label activity events using insiders.csv time windows

In [ ]:
# Cell 8: label activity events using insiders.csv time windows

insiders_df = pd.read_csv("data/raw/insiders.csv")
insiders_df["start"] = pd.to_datetime(insiders_df["start"], errors="coerce")
insiders_df["end"] = pd.to_datetime(insiders_df["end"], errors="coerce")

activity_df["date"] = pd.to_datetime(activity_df["date"], errors="coerce")

# one insider window per user for r6.2-1..5
insider_windows = insiders_df[insiders_df["user"].isin(activity_df["insider_user"].unique())].copy()

print("Insider windows used:", insider_windows.shape)
print(insider_windows[["user", "start", "end"]].head())

# merge each event with its insider window by insider_user
activity_labeled = activity_df.merge(
    insider_windows[["user", "start", "end"]],
    left_on="insider_user",
    right_on="user",
    how="left",
    suffixes=("", "_insider")
)

activity_labeled["is_insider"] = (
    (activity_labeled["user"] == activity_labeled["insider_user"]) &
    (activity_labeled["date"] >= activity_labeled["start"]) &
    (activity_labeled["date"] <= activity_labeled["end"])
).astype("int8")

print("Labeled activity shape:", activity_labeled.shape)
print(activity_labeled[["activity_type", "user", "date", "insider_user", "is_insider"]].head(10))
print("Label counts:")
print(activity_labeled["is_insider"].value_counts(dropna=False))

Cell 9: Aggregate labeled events to user-day and merge into user_day_df

In [ ]:
# Cell 9: aggregate labeled events to user-day and merge into user_day_df

activity_labeled["day"] = activity_labeled["date"].dt.normalize()

user_day_labels = (
    activity_labeled
    .groupby(["insider_user", "day"], as_index=False)["is_insider"]
    .max()
    .rename(columns={"insider_user": "user"})
)

print("User-day labels shape:", user_day_labels.shape)
print(user_day_labels.head())
print(user_day_labels["is_insider"].value_counts())

# merge labels into the existing feature table
user_day_df["day"] = pd.to_datetime(user_day_df["day"], errors="coerce")

labeled_user_day_df = user_day_df.merge(
    user_day_labels,
    on=["user", "day"],
    how="left"
)

labeled_user_day_df["is_insider"] = labeled_user_day_df["is_insider"].fillna(0).astype("int8")

print("Labeled user-day shape:", labeled_user_day_df.shape)
print(labeled_user_day_df[["user", "day", "is_insider"]].head(10))
print(labeled_user_day_df["is_insider"].value_counts())

In [ ]:
processed_path = os.path.join("data", "processed")

# Parquet path
labeled_user_day_parquet_path = os.path.join(processed_path, "labeled_user_day_df.parquet")
labeled_user_day_df.to_parquet(labeled_user_day_parquet_path, index=False)

activity_labeled_parquet_path = os.path.join(processed_path, "activity_labeled.parquet")
activity_labeled.to_parquet(activity_labeled_parquet_path, index=False)

# CSV path
labeled_user_day_csv_path = os.path.join(processed_path, "labeled_user_day_df.csv")
labeled_user_day_df.to_csv(labeled_user_day_csv_path, index=False)

activity_labeled_csv_path = os.path.join(processed_path, "activity_labeled.csv")
activity_labeled.to_csv(activity_labeled_csv_path, index=False)

Cell 10: Rough Works

In [ ]:
print(labeled_user_day_df.columns.tolist())